# Experiment 12: ordinary vs over-forecasting LightGBM

A controlled chronological comparison between standard RMSE-trained LightGBM and a 0.60-quantile LightGBM. Both use identical features and maximum boosting rounds; only the training objective differs.

In [1]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root=os.environ.get('KDDM_PROJECT_ROOT'); roots=([Path(configured_root)] if configured_root else [])+[Path.cwd(),*Path.cwd().parents]
project_root=next((p for p in roots if (p/'exploration/metul/preprocessing.py').exists()),None)
if project_root is None: raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir=project_root/'exploration'/'metul'; sys.path.insert(0,str(experiment_dir))
from preprocessing import RentalPreprocessor, load_datasets
TARGET='MONTHLY_RENT'; QUANTILE=0.60; TIME_LIMIT=int(os.environ.get('QUANTILE_LGBM_TIME_LIMIT','600')); RUN_ID=os.environ.get('AUTOGLUON_RUN_ID',pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
FEATURES=['TOWN','BLOCK','STREET','FLAT_TYPE','FLAT_MODEL','FLOOR_AREA_SQM','approval_date_num','approval_month','coe_quota_premium_mean','YEAR_COMPLETED','MAX_FLOOR','mrt_dist_nearest_km','mall_dist_nearest_km']
_,datasets=load_datasets(project_root/'data'); raw=datasets['train'].copy(); raw['period']=pd.to_datetime(raw.RENT_APPROVAL_DATE+'-01').dt.to_period('M'); periods=np.sort(raw.period.unique()); cutoff=periods[int(len(periods)*.80)]; fit_raw,valid_raw=raw[raw.period<cutoff].copy(),raw[raw.period>=cutoff].copy()
preprocessor=RentalPreprocessor(datasets); fit_x=preprocessor.fit_transform(fit_raw.drop(columns=['period',TARGET])); valid_x=preprocessor.transform(valid_raw.drop(columns=['period',TARGET])); fit_x,valid_x=fit_x[FEATURES],valid_x[FEATURES]; fit_y,valid_y=fit_raw[TARGET].astype(float).reset_index(drop=True),valid_raw[TARGET].astype(float).reset_index(drop=True)
print(f'Cutoff={cutoff}; fit={len(fit_x):,}; validation={len(valid_x):,}; over-forecast quantile={QUANTILE:.2f}')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [2]:
LGBM_PARAMS={'num_leaves':63,'learning_rate':.05,'num_boost_round':1800,'min_data_in_leaf':25,'feature_fraction':.9}
normal=TabularPredictor(label=TARGET,problem_type='regression',eval_metric='root_mean_squared_error',path=str(experiment_dir/'ag_exp_12_quantile'/RUN_ID/'ordinary_lgbm')).fit(train_data=fit_x.assign(**{TARGET:fit_y.to_numpy()}),hyperparameters={'GBM':LGBM_PARAMS},fit_weighted_ensemble=False,presets=['medium_quality','ignore_text'],time_limit=TIME_LIMIT,verbosity=2)
quantile=TabularPredictor(label=TARGET,problem_type='quantile',quantile_levels=[QUANTILE],eval_metric='pinball_loss',path=str(experiment_dir/'ag_exp_12_quantile'/RUN_ID/'quantile_lgbm')).fit(train_data=fit_x.assign(**{TARGET:fit_y.to_numpy()}),hyperparameters={'GBM':LGBM_PARAMS},fit_weighted_ensemble=False,presets=['medium_quality','ignore_text'],time_limit=TIME_LIMIT,verbosity=2)
normal_prediction=normal.predict(valid_x).astype(float).to_numpy(); quantile_prediction=quantile.predict(valid_x)[QUANTILE].astype(float).to_numpy()
def report(name,prediction):
    residual=prediction-valid_y.to_numpy(); under=residual<0
    return {'model':name,'RMSE':mean_squared_error(valid_y,prediction)**.5,'MAE':mean_absolute_error(valid_y,prediction),'R2':r2_score(valid_y,prediction),'mean_signed_error':residual.mean(),'underforecast_rate':under.mean(),'mean_underforecast_amount':(-residual[under]).mean(),'overforecast_rate':(~under).mean(),'mean_overforecast_amount':residual[~under].mean()}
comparison=pd.DataFrame([report('Ordinary LGBM (RMSE)',normal_prediction),report(f'Quantile LGBM (q={QUANTILE:.2f})',quantile_prediction)])
display(comparison)
predictions=pd.DataFrame({'actual':valid_y,'ordinary_prediction':normal_prediction,'quantile_prediction':quantile_prediction}); predictions['quantile_minus_ordinary']=predictions.quantile_prediction-predictions.ordinary_prediction
display(predictions[['quantile_minus_ordinary']].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))
print(f'Expected calibrated q={QUANTILE:.2f} behavior: approximately {QUANTILE:.0%} of validation rents should be at or below the quantile prediction.')


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.71 GB / 16.00 GB (35.7%)
Disk Space Avail:   176.35 GB / 460.38 GB (38.3%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_12_quantile/20260929_212110/ordinary_lgbm"
Train Data Rows:    117988
Train Data Columns: 13
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to p

,model,RMSE,MAE,R2,mean_signed_error,underforecast_rate,mean_underforecast_amount,overforecast_rate,mean_overforecast_amount
0,Ordinary LGBM (RMSE),478.369587,356.437704,0.469840,-51.787650,0.612395,333.302133,0.387605,392.990713
1,Quantile LGBM (q=0.60),492.281263,343.086805,0.438556,124.119267,0.423654,258.427548,0.576346,405.317094


,quantile_minus_ordinary
count,32012.000000
mean,175.906917
std,105.836633
min,-468.873047
1%,-71.807227
5%,9.174841
25%,108.737976
50%,173.544922
75%,239.828613
95%,352.082593


Expected calibrated q=0.60 behavior: approximately 60% of validation rents should be at or below the quantile prediction.
